# 05 — Fee Benchmarks (Seller OS)

Every channel taxes you differently. This notebook pins **per-channel fee tables**, the **demo blended rate ($16.71 / $274 = 6.1%)**, the **12% creep threshold**, and **3 shop examples**. All math is stdlib arithmetic — no repo imports.

Outputs below are **pre-filled from a real run**.

## 0. The one formula

`fee = round(gmv * rate + fixed, 2)` and `blended = total_fees / GMV * 100`. If blended >= 12%, you have `FEE_CREEP`.

In [1]:
def fee_shopify(gmv):
    return round(gmv*0.029+0.30, 2)

def fee_etsy(gmv):
    return round(gmv*0.065+0.20, 2)

def fee_amazon(gmv):
    return round(gmv*0.15, 2)

def fee_tiktok(gmv):
    return round(gmv*0.05+0.30, 2)

def fee_ebay(gmv):
    return round(gmv*0.1325+0.30, 2)

print('fee fns ok: shopify/etsy/amazon/tiktok/ebay')

fee fns ok: shopify/etsy/amazon/tiktok/ebay


### Per-channel fee table — $34.25 ticket (demo AOV)

| channel | formula | fee | take-rate |
|---|---|---|---|
| shopify | 2.9% + $0.30 | $1.29 | 3.8% |
| etsy | 6.5% + $0.20 | $2.43 | 7.1% |
| amazon | 15% | $5.14 | 15.0% |
| tiktok | 5% + $0.30 | $2.01 | 5.9% |
| ebay | 13.25% + $0.30 | $4.84 | 14.1% |

In [2]:
aov = 34.25
for name, fn in [('shopify', fee_shopify), ('etsy', fee_etsy), ('amazon', fee_amazon), ('tiktok', fee_tiktok), ('ebay', fee_ebay)]:
    f = fn(aov)
    print(f"{name:8} fee=${f:.2f} take={f/aov*100:.1f}%")

ERROR: name 'fee_shopify' is not defined


### Shopify 2.9% + $0.30 — small tickets hurt most

$0.30 fixed dominates a $15 sale (4.9%) but fades at $100 (3.2%).

In [3]:
for gmv in (15.00, 34.25, 100.00):
    f = fee_shopify(gmv)
    print(f"shopify ${gmv:.2f} -> fee ${f:.2f} ({f/gmv*100:.1f}%)")

ERROR: name 'fee_shopify' is not defined


### Etsy 6.5% + $0.20 — ~2x Shopify

Etsy funds search/ads with the extra ~3.6pp. Worth it only if conversion covers it.

In [4]:
for gmv in (15.00, 34.25, 100.00):
    f = fee_etsy(gmv)
    print(f"etsy ${gmv:.2f} -> fee ${f:.2f} ({f/gmv*100:.1f}%)")

ERROR: name 'fee_etsy' is not defined


### Amazon 15% — no fixed fee, brutal on margin

15% with no fixed leg: flat 15.0% at every ticket. Needs 2x volume to justify.

In [5]:
for gmv in (15.00, 34.25, 100.00):
    f = fee_amazon(gmv)
    print(f"amazon ${gmv:.2f} -> fee ${f:.2f} ({f/gmv*100:.1f}%)")

ERROR: name 'fee_amazon' is not defined


### TikTok Shop 5% + $0.30 — between Shopify and Etsy

Promo-era rates vary; 5% + 30c is the planning baseline used here.

In [6]:
for gmv in (15.00, 34.25, 100.00):
    f = fee_tiktok(gmv)
    print(f"tiktok ${gmv:.2f} -> fee ${f:.2f} ({f/gmv*100:.1f}%)")

ERROR: name 'fee_tiktok' is not defined


### eBay 13.25% + $0.30 — near-Amazon

13.25% + 30c lands at ~14-15%. Final-value fees include shipping in reality; here GMV-only.

In [7]:
for gmv in (15.00, 34.25, 100.00):
    f = fee_ebay(gmv)
    print(f"ebay ${gmv:.2f} -> fee ${f:.2f} ({f/gmv*100:.1f}%)")

ERROR: name 'fee_ebay' is not defined


### Stripe / PayPal / Square variants (direct payments)

| rail | formula | $34.25 fee |
|---|---|---|
| stripe | 2.9% + 30c | $1.29 |
| paypal | 3.49% + 49c | $1.69 |
| square | 2.6% + 10c | $0.99 |

In [8]:
def fee_stripe(g): return round(g*0.029+0.30, 2)
def fee_paypal(g): return round(g*0.0349+0.49, 2)
def fee_square(g): return round(g*0.026+0.10, 2)
aov = 34.25
print(f"stripe ${fee_stripe(aov):.2f}")
print(f"paypal ${fee_paypal(aov):.2f}")
print(f"square ${fee_square(aov):.2f}")

stripe $1.29
paypal $1.69
square $0.99


## 1. Worked demo math — fees $16.71 on $274 = 6.1% HEALTHY

Demo: 5 Shopify rows (estimated 2.9%+30c) + 3 Etsy rows (explicit) = $16.71. Blended 6.1% < 12% threshold, so **no FEE_CREEP**.

In [9]:
gmv, fees = 274.00, 16.71
blended = fees/gmv*100
thresh = 12.0
print(f"blended {fees}/{gmv} = {blended:.1f}%")
print(f"threshold {thresh:.1f}% = ${gmv*thresh/100:.2f}")
print(f"headroom ${gmv*thresh/100-fees:.2f} -> HEALTHY" if blended < thresh else 'CREEP')

blended 16.71/274.0 = 6.1%
threshold 12.0% = $32.88
headroom $16.17 -> HEALTHY


### Blended-rate check — headroom in dollars

12% of $274 = $32.88. Demo pays $16.71, so **$16.17 headroom** before creep. One Amazon migration (+~9pp) would eat it.

In [10]:
gmv, fees = 274.00, 16.71
cap = round(gmv*0.12, 2)
print(f"cap ${cap:.2f} paid ${fees:.2f} headroom ${cap-fees:.2f}")
print(f"amazon-only est ${round(gmv*0.15,2):.2f} -> {(round(gmv*0.15,2)/gmv*100):.1f}% CREEP")

cap $32.88 paid $16.71 headroom $16.17
amazon-only est $41.10 -> 15.0% CREEP


## 2. Three shop examples — healthy / creep / mixed

- **A healthy**: Shopify-only, blended ~3.6%.
- **B creep (feebleed)**: inflated 18.5% + $1.20/row, blended ~20%.
- **C mixed**: Shopify + Etsy + Amazon, blended ~10% (watch).

In [11]:
shops = [('A-healthy', 462.50, 16.52), ('B-creep', 526.00, 105.20), ('C-mixed', 600.00, 60.00)]
for name, g, f in shops:
    flag = 'HEALTHY' if f/g*100 < 12 else 'FEE_CREEP'
    print(f"{name} gmv=${g:.2f} fees=${f:.2f} blended={f/g*100:.1f}% -> {flag}")

A-healthy gmv=$462.50 fees=$16.52 blended=3.6% -> HEALTHY
B-creep gmv=$526.00 fees=$105.20 blended=20.0% -> FEE_CREEP
C-mixed gmv=$600.00 fees=$60.00 blended=10.0% -> HEALTHY


### Shop A — healthy (mirrors `shop_01`)

GMV $462.50, fees $16.52 → 3.6%. All-Shopify, small fixed-fee drag. Action: push best seller, no fee work.

In [12]:
g, f = 462.50, 16.52
print(f"A blended={f/g*100:.1f}% headroom=${round(g*0.12-f,2):.2f}")
print('action: no fee work; push best seller')

A blended=3.6% headroom=$38.98
action: no fee work; push best seller


### Shop B — creep (mirrors `shop_04` feebleed)

GMV $526.00, fees $105.20 → 20.0%. Formula 18.5% + $1.20/row. Action: renegotiate plan, move volume to Shopify, raise prices 8-10%.

In [13]:
g, f = 526.00, 105.20
print(f"B blended={f/g*100:.1f}% over-cap=${round(f-g*0.12,2):.2f}")
print('action: FEE_CREEP -> renegotiate + reprice')

B blended=20.0% over-cap=$42.08
action: FEE_CREEP -> renegotiate + reprice


### Shop C — mixed (watch band 8-12%)

GMV $600.00, fees $60.00 → 10.0%. One bad month (more Amazon) tips it over. Action: cap Amazon share <25%, review monthly.

- [ ] Compute blended monthly (fees/GMV)
- [ ] Alert at >=12% (FEE_CREEP)
- [ ] Compare channel mix before repricing
- [ ] Keep this table pinned at packing desk

In [14]:
g, f = 600.00, 60.00
print(f"C blended={f/g*100:.1f}% headroom=${round(g*0.12-f,2):.2f} -> WATCH")
print('rule: amazon_share<25% or blended drifts +2pp')

C blended=10.0% headroom=$12.00 -> WATCH
rule: amazon_share<25% or blended drifts +2pp
